# 02 — Leakage-controlled borrower features

Historical tables are aggregated without the target and without fitting any population distribution. Learned clipping, imputation, rare-category pooling, encoding, and scaling are deferred to model folds in notebook 03.

In [1]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.features import build_feature_table, feature_family_summary, validate_feature_table

RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"
TABLES = ROOT / "outputs" / "tables"
PROCESSED.mkdir(parents=True, exist_ok=True)
TABLES.mkdir(parents=True, exist_ok=True)

In [2]:
features, temporal_audit = build_feature_table(RAW)
validate_feature_table(features, 307_511)
families = feature_family_summary(features)
features.to_pickle(PROCESSED / "borrower_features.pkl")
families.to_csv(TABLES / "feature_family_summary.csv", index=False)

print(f"Borrowers: {len(features):,}")
print(f"Candidate predictors: {features.shape[1] - 2}")
display(families)
display(temporal_audit)

Borrowers: 307,511
Candidate predictors: 98


,feature_family,prefix,feature_count
0,Application,APP_,23
1,Bureau,BUREAU_,19
2,Bureau balance,BB_,12
3,Previous applications,PREV_,22
4,Installments,INST_,22


,source,input_rows,retained_rows,excluded_rows,rule,temporal_excluded_rows,unmapped_bureau_rows,temporal_or_missing_date_excluded_rows,outside_application_population_rows
0,bureau,1465325,1465308,17,DAYS_CREDIT <= 0 and DAYS_CREDIT_UPDATE <= 0,NaN,NaN,NaN,NaN
1,bureau_balance,27299925,14701510,12598415,MONTHS_BALANCE <= 0 and SK_ID_BUREAU maps to a...,0.0,12598415.0,NaN,NaN
2,previous_application,1413701,1413701,0,DAYS_DECISION < 0,NaN,NaN,NaN,NaN
3,installments_payments,13605401,11589009,2016392,borrower in application_train; both scheduled ...,NaN,NaN,2905.0,2013487.0


In [3]:
assert features["SK_ID_CURR"].is_unique
assert features["TARGET"].notna().all()
assert 60 <= features.shape[1] - 2 <= 150
assert temporal_audit.loc[temporal_audit["source"].eq("bureau"), "excluded_rows"].iloc[0] == 17
assert temporal_audit.loc[temporal_audit["source"].eq("installments_payments"), "temporal_or_missing_date_excluded_rows"].iloc[0] == 2_905

The candidate set is intentionally restrained. It excludes `CODE_GENDER`, opaque `EXT_SOURCE` scores, raw `AMT_GOODS_PRICE` (near-collinear with credit), future-realization lifecycle fields, and generic min/max/mean aggregation grids. `C` and `X` bureau-balance statuses are not treated as delinquency severities. Explicit no-history indicators distinguish absence of an observed relationship from an ordinary missing value.